## Enhancing Functions: Decorators

- A **decorator** is a callable that takes another function, adds behaviour before and/or after it runs, and returns a new callable.  
- They solve cross‑cutting concerns such as logging, timing, permission checks, or retries without cluttering core logic.  
- The magic `@decorator_name` syntax is shorthand for passing the target function to the decorator and re‑binding the original name to the returned wrapper.

## Decorator Anatomy (Manual View)

- **Outer decorator function** accepts the target function and creates a **wrapper** inside it.  
- The wrapper usually takes `*args, **kwargs` so it can handle any signature.  
- Wrapper executes optional "before" code, calls the original, maybe does "after" code, and returns the original’s result.  
- Returning the wrapper from the decorator completes the transformation.

Using decorators:
- Manually wrapping illustrates what `@` syntax really does behind the scenes.
- This approach is clear but repetitive: `@` eliminates the manual reassignment step.  

In [8]:
import time

def simple_print(sleep_duration):
    time.sleep(sleep_duration)
    print("Running the original simple function")

# Here the way to preserve the original function state, even altered  by wrapper function.
original_simple_func = simple_print

def timing_decorator(original_func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = original_func(*args, **kwargs)
        duration = time.perf_counter() - start
        print(f"{original_func.__name__} took {duration:.3f}s")
        return result
    return wrapper

simple_print = timing_decorator(simple_print)
print(simple_print)
simple_print(0.1)
simple_print(0.4)

# Here is the way to preserve the original function state, even altered  by wrapper function.
print("Retriving the original function:...")
original_via_closure = simple_print.__closure__[0].cell_contents
print(original_via_closure)

original_via_closure(0.5)
original_simple_func(0.6)

<function timing_decorator.<locals>.wrapper at 0x000001A558321A60>
Running the original simple function
simple_print took 0.101s
Running the original simple function
simple_print took 0.401s
Retriving the original function:...
<function simple_print at 0x000001A559976A30>
Running the original simple function
Running the original simple function


## The `@` Syntax

- Placing `@decorator_name` directly above `def my_func():` triggers `my_func = decorator_name(my_func)` at *definition* time.  
- After that line is executed, `my_func` refers to the wrapper returned by the decorator, so callers automatically get enhanced behaviour.  
- This keeps the decoration visible and close to the function definition, improving readability.  

In [24]:
import dis

@timing_decorator
def another_task():
    print("Running another task...")

another_task()
simple_print(0.7)
original_via_an_closure = another_task.__closure__[0].cell_contents
original_via_an_closure()

print(simple_print.__code__.co_code)

dis.dis(simple_print.__closure__[0].cell_contents)

Running another task...
another_task took 0.000s
Running the original simple function
simple_print took 0.701s
Running another task...
b'<\x01\x80\x00\\\x00\x00\x00\x00\x00\x00\x00\x00\x00P\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x004\x00\x00\x00\x00\x00\x00\x00p\x02S\x05!\x00V\x00/\x00V\x01B\x01\x04\x00p\x03\\\x00\x00\x00\x00\x00\x00\x00\x00\x00P\x03\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x004\x00\x00\x00\x00\x00\x00\x00V\x02,\n\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00p\x04\\\x05\x00\x00\x00\x00\x00\x00\x00\x00S\x05P\x06\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0c\x00R\x00V\x04R\x01\r\x00R\x022\x044\x01\x00\x00\x00\x00\x00\x00\x1f\x00V\x03#\x00'
  3           RESUME                   0

  4           LOAD_GLOBAL              0 (time)
              LOAD_ATTR                3 (sleep + NULL|self)
              LOAD_FAST_BORROW         0 (sleep_duration)
              CALL                     1
   

In [39]:
def outer():
    x = 10
    y = 20
    def inner():
        print(x, y)   # captures TWO variables
    return inner

fn = outer()

print(fn.__closure__)
# (<cell at 0x...>, <cell at 0x...>)   ← tuple of 2 cells

print(fn.__closure__[0].cell_contents)  # → 10  (x)
print(fn.__closure__[1].cell_contents)  # → 20  (y)

print(outer.__code__.co_consts)
dis.dis(outer)

print("Outer Large started ......")
def outer_large():
    x = 999999
    y = 999991
    z = 999997
    def inner():
        print(x)
    return inner

print("Large Int constants:", outer_large.__code__.co_consts)
dis.dis(outer_large)

# NOTE
# 1. The Small Int Scenario (10, 20)
# Bytecode: LOAD_SMALL_INT 20

# Why it was "missing" from co_consts: The value 20 was baked directly into the instruction itself. 
# Python 3.11+ does this for numbers typically between -1 and 256. 
# It's faster because the CPU doesn't have to look up a value in a separate table.

# Closure: Even though it was "missing" from the constants, 
#          it correctly appeared in the closure because STORE_DEREF moved that inlined value into the "cell" for the inner function.

# 2. The Large Int Scenario (999999)
# Bytecode: LOAD_CONST 0 (999999)

# The co_consts output: (999999, 999991, 999997, <code object inner...>)

# Observation: Notice that 999999, 999991, and 999997 are all explicitly listed in the constants tuple. 
#              Because these numbers are too large to be "inlined," Python had to put them in the "storage bin" (co_consts) 
#              and use a pointer to grab them.

(<cell at 0x000001A5585DFA90: int object at 0x00007FF9E2C77598>, <cell at 0x000001A5585DC640: int object at 0x00007FF9E2C776D8>)
10
20
(10, <code object inner at 0x000001A558343D20, file "C:\Users\islam\AppData\Local\Temp\ipykernel_25096\128651042.py", line 4>)
  --           MAKE_CELL                1 (x)
               MAKE_CELL                2 (y)

   1           RESUME                   0

   2           LOAD_SMALL_INT          10
               STORE_DEREF              1 (x)

   3           LOAD_SMALL_INT          20
               STORE_DEREF              2 (y)

   4           LOAD_FAST_BORROW         1 (x)
               LOAD_FAST_BORROW         2 (y)
               BUILD_TUPLE              2
               LOAD_CONST               1 (<code object inner at 0x000001A558343D20, file "C:\Users\islam\AppData\Local\Temp\ipykernel_25096\128651042.py", line 4>)
               MAKE_FUNCTION
               SET_FUNCTION_ATTRIBUTE   8 (closure)
               STORE_FAST               0 (i